## The transaction log & ACID — how a commit buys the guarantees

A **Delta table** on disk is two things in one directory: the **Parquet data files**, and a `_delta_log` directory beside them. Anything that reads Parquet can read the data; the log is what turns that pile of files into a table.

The `_delta_log` holds an ordered sequence of JSON commit files, one per transaction — `0000...000.json`, `...001.json`, and so on. Each commit lists **actions**:

- **`add`** — a Parquet file is now part of the table. It carries `stats` (min/max/null-count per column) so Delta can skip files at read time without opening them.
- **`remove`** — a file is no longer part of the table. The bytes aren't deleted yet; `VACUUM` does that later.
- **`metaData`** — the schema, partitioning, and properties. Schema changes emit a new `metaData`.

A reader replays the log to compose the current state: which files belong, and what columns they have. Every ten commits Delta writes a **checkpoint** (a Parquet snapshot of the whole log) so reads stay fast as history grows.

Those atomic JSON commits are exactly what buy **ACID** on object storage:

- **Atomicity** — a commit lands when the next `.json` appears (a single atomic PUT). Readers see it or they don't; data files written *before* it aren't part of the table because no `add` references them.
- **Consistency** — every commit is validated against the current schema and constraints.
- **Isolation** — **optimistic concurrency**: two writers prepare independently; whoever writes their `.json` first wins, the loser re-reads and retries.
- **Durability** — the object store persists the data and log files.

So the bank's nightly pipeline and the real-time fraud stream can write the same table without corrupting each other — each commit either fully succeeds or retries, never a partial result.